# Live CAT Sessions

**Audience:** application developers who run adaptive tests in a product. **Level:** intermediate.
You know Python; the [introduction](https://douglasrizzo.com.br/catsim/introduction.html) covers the IRT this notebook needs, and
[Terms and notation](https://douglasrizzo.com.br/catsim/terms-and-notation.html) the terms it uses.

In an application, catsim decides which item to show next and when the test ends, and the
application shows the items, collects the answers and stores the sessions. In this notebook you will:

- run sessions with a response provider, one step at a time and whole;
- read each step's result, why a session ended and which criterion ended it;
- share item exposure across sessions, and decline overexposed items with an exposure control;
- save a session and the exposure tracker in the middle of a test, restart, and resume.

It is the hands-on companion of the [Live CAT sessions](https://douglasrizzo.com.br/catsim/live-sessions.html) guide, which explains each step
in full. To run it outside this repository, install catsim first: `pip install catsim`.

In [ ]:
import dataclasses
import json

import numpy
from typing_extensions import override

from catsim import (
  CatEngine,
  CatSessionSnapshot,
  CatSessionState,
  Estimate,
  ExposureTracker,
  ItemBank,
  ResponseProvider,
  RunContext,
  SimulatedResponseProvider,
  StepContext,
)
from catsim.estimation import EAPEstimator
from catsim.exposure import BaseExposureControl
from catsim.initialization import FixedPointInitializer
from catsim.selection import MaxInfoSelector
from catsim.stopping import MaxLengthStopper, StandardErrorStopper, StopRule

## Build the Engine

A `CatEngine` holds the item bank and the four strategies of a test. A generated bank of 200 items
under the three-parameter logistic model stands in for your calibrated one. The test starts at an
ability estimate of 0, gives the most informative item at the current estimate among those whose
exposure rate is below `R_MAX` (0.3), estimates ability by expected a posteriori (EAP), and stops
once the standard error is at most `MAX_SE` (0.4). The cap of `MAX_ITEMS` (20) ends the test
when the bank cannot reach that precision. Build the engine once and share it: it keeps nothing
about any session.

The `RunContext` holds what the sessions share: the random generator the strategies draw from, and
the `ExposureTracker` that counts how often each item is given. Two seeds make this notebook
repeat: `BANK_SEED` draws the bank and `RUN_SEED` seeds the run's generator. An application
creates its generator unseeded, with `numpy.random.default_rng()`.

In [ ]:
BANK_SEED = 7
RUN_SEED = 2024
N_ITEMS = 200
R_MAX = 0.3
MAX_SE = 0.4
MAX_ITEMS = 20

item_bank = ItemBank.generate(N_ITEMS, "3PL", rng=BANK_SEED)
stopper = StopRule(StandardErrorStopper(MAX_SE), MaxLengthStopper(MAX_ITEMS))
engine = CatEngine(item_bank, FixedPointInitializer(0.0), MaxInfoSelector(r_max=R_MAX), EAPEstimator(), stopper)
tracker = ExposureTracker(item_bank.n_items)
context = RunContext(rng=numpy.random.default_rng(RUN_SEED), exposure_tracker=tracker)

## Step a Session with a Response Provider

When the application gets each answer within the call, `engine.step` selects an item, asks a
response provider for the answer and applies it. A provider is any object with an `answer`
method; subclassing `ResponseProvider` lets a type checker check its signature. `ReplayedAnswers`
replays answers recorded in order: the first answer goes to the first item given, the second to
the second, and so on. `ANSWERS` holds 20 of them, one for each item of the longest test.

Each step returns a `CatStepResult`: the item given, the response, and a snapshot of the session
after the step, with its ability estimate and standard error.

In [ ]:
class ReplayedAnswers(ResponseProvider):
  """Answer each item with the next of a list of recorded answers."""

  def __init__(self, answers: list[bool]) -> None:
    self.answers = answers

  @override
  def answer(self, item_bank: ItemBank, item_id: int, session: CatSessionSnapshot, rng: numpy.random.Generator) -> bool:
    return self.answers[len(session.responses)]


ANSWERS = [True, True, False, True, False] * 4
replay = ReplayedAnswers(ANSWERS)

first = engine.start_session("attempt-1", context)
steps = []
while not first.is_stopped:
  step = engine.step(first, replay, context)
  steps.append(step)
  outcome = "correct" if step.response else "incorrect"
  print(
    f"item {step.selected_item_id:3}: {outcome:9} estimate {step.current_theta:5.2f}, "
    f"standard error {step.session.current_se:.3f}"
  )
print(step.stopped, step.stop_reason)

Each correct answer raises the estimate and each incorrect one lowers it, by smaller steps as the
test goes on, and the standard error shrinks with almost every item. The seventeenth step brings
the standard error to 0.393, at most `MAX_SE`, so that step reports the session stopped, with
the stop reason `stopper`: the stopper ended the test.

Each result is a snapshot: it keeps the session as its step left it. The stopper is a `StopRule`
of two criteria; asked about a session, with the context `engine.context` returns for it, the
rule tells which of them hold:

In [ ]:
print(len(steps[0].session.administered_items), first.administered_count)
print(stopper.satisfied(engine.context(first)))

The first step's snapshot still holds one item, while the session holds 17. The precision
criterion ended the test, before the length cap.

## Run a Whole Session

`engine.run_session` repeats `step` until the session ends. A second examinee gives the same
recorded answers:

In [ ]:
second = engine.run_session(engine.start_session("attempt-2", context), replay, context)
print(set(first.administered_items) & set(second.administered_items))
print(second.administered_count, second.stop_reason, f"{second.current_se:.3f}")
print(stopper.satisfied(engine.context(second)))
print(engine.step(second, replay, context).selected_item_id)

The two sessions share no item. For `attempt-2`, every item `attempt-1` was given has an exposure
rate above the cap of 0.3, so the selector passes over them. The items left are less informative
at the examinee's estimate: after 20 items the standard error is still 0.409, and the length cap
ends the test. A session that has ended takes no more steps: `step` returns a result with no
item, `None`. The other stop reason, `no_items`, ends a session whose selector has no item left
to offer.

## Share Exposure across Sessions

Every session on the item bank shares the tracker of the run context, so the cap sees what all of
them were given. Below, 50 simulated examinees, with abilities evenly spread from -2 to 2, stand
in for a day of real ones: `SimulatedResponseProvider` draws each answer from the examinee's true
ability with the run's generator.

In [ ]:
N_EXAMINEES = 50

for j, true_theta in enumerate(numpy.linspace(-2, 2, N_EXAMINEES)):
  session = engine.start_session(f"examinee-{j}", context, true_theta=true_theta)
  engine.run_session(session, SimulatedResponseProvider(), context)
print(tracker.n_sessions, numpy.count_nonzero(tracker.counts), numpy.max(tracker.counts))

The tracker has counted 52 sessions, the two attempts and the 50 examinees. They were given 115
of the 200 items, and no item went to more than 16 of them, about 0.3 of the sessions: the cap
spreads the administrations over the bank. Without a tracker in the run context, the selector
gets no exposure rates and applies no cap.

## Decline Items with an Exposure Control

The selector's cap passes over an item only while other items remain: once every remaining item
has reached the cap, it gives one anyway. An exposure control decides after the selector, for any
selector: the engine asks it about each item the selector proposes, and an item it declines is
excluded from the session for the rest of its test. `HardExposureCap` declines every item whose
exposure rate has reached `R_MAX`, so it never gives a capped item; a session whose remaining
items are all declined ends with the stop reason `no_items`. It holds its cap and nothing about
any session, as every strategy must ([Extending catsim](https://douglasrizzo.com.br/catsim/extending.html)).

In [ ]:
class HardExposureCap(BaseExposureControl):
  """Decline every item whose exposure rate has reached the cap."""

  def __init__(self, r_max: float) -> None:
    self.r_max = r_max

  @override
  def accept(self, context: StepContext, item_id: int) -> bool:
    return context.exposure_rates is None or bool(context.exposure_rates[item_id] < self.r_max)


capped = CatEngine(
  item_bank,
  FixedPointInitializer(0.0),
  MaxInfoSelector(),
  EAPEstimator(),
  stopper,
  exposure_control=HardExposureCap(R_MAX),
)

## Save and Resume a Session

In a web application, `select_next` and `apply_response` run in different requests: one shows
the item, the next receives the answer. Save the session, the item on screen and the tracker
after every call, as [the guide](https://douglasrizzo.com.br/catsim/live-sessions.html#save-and-resume-a-session) explains. Here a dictionary
stands in for the database, and `next_item` and `answer` for the two requests. Like an
application's handlers, they use the engine, the run context and the tracker the process holds,
which the restart below rebuilds.

In [ ]:
database: dict[str, str] = {}


def save(session: CatSessionState, shown: int | None) -> None:
  """Store the session, the item on screen and the tracker, as one transaction would."""
  database[session.session_id] = json.dumps({"session": dataclasses.asdict(session), "shown": shown})
  counts = {"counts": tracker.counts.tolist(), "proposals": tracker.proposals.tolist()}
  database["tracker"] = json.dumps({**counts, "n_sessions": tracker.n_sessions})


def load(session_id: str) -> tuple[CatSessionState, int | None]:
  """Rebuild a stored session and return it with the item on screen."""
  row = json.loads(database[session_id])
  fields = row["session"]
  estimates = [Estimate(**estimate) for estimate in fields["estimates"]]
  return CatSessionState(**{**fields, "estimates": estimates}), row["shown"]


def next_item(session: CatSessionState) -> int | None:
  """Handle the request that shows an item: select it, and save the session with it on screen."""
  item_id = capped.select_next(session, context)
  save(session, item_id)
  return item_id


def answer(session: CatSessionState, item_id: int, *, response: bool) -> None:
  """Handle the request that receives the answer: apply it, and save the session."""
  capped.apply_response(session, item_id, response=response, context=context)
  save(session, None)

A third examinee answers three items, and the process stops while the fourth is on screen:

In [ ]:
third = capped.start_session("attempt-3", context)
save(third, None)
for recorded in ANSWERS[:3]:
  answer(third, next_item(third), response=recorded)
shown = next_item(third)
print(third.administered_items, shown, third.excluded_items)
print(database["attempt-3"])

By the time the fourth item is on screen, the control has declined five: 51, 173, 164, 187 and
180, each the most informative item at the examinee's estimate when it was proposed, and each at
an exposure rate that had reached the cap of 0.3. They are in the saved `excluded_items`, and the
tracker counts them among its `proposals`. The initial estimate's standard error is saved as
`Infinity`, which strict JSON parsers reject ([the guide](https://douglasrizzo.com.br/catsim/live-sessions.html#save-and-resume-a-session)
says how to store it).

After a restart, the application rebuilds the tracker and the run context from storage, loads
the session, and passes the answer to the item on screen to `apply_response`, without selecting
again. The engine holds nothing about any session, so the restarted process builds the same one.

In [ ]:
restarted = ExposureTracker(item_bank.n_items, **json.loads(database["tracker"]))
print(
  numpy.array_equal(restarted.counts, tracker.counts),
  numpy.array_equal(restarted.proposals, tracker.proposals),
  restarted.n_sessions == tracker.n_sessions,
)
tracker = restarted
context = RunContext(rng=numpy.random.default_rng(), exposure_tracker=tracker)

resumed, shown = load("attempt-3")
print(resumed == third, shown)
answer(resumed, shown, response=ANSWERS[resumed.administered_count])
while (item_id := next_item(resumed)) is not None:
  answer(resumed, item_id, response=ANSWERS[resumed.administered_count])
print(resumed.administered_count, resumed.stop_reason, len(resumed.excluded_items))

The rebuilt tracker and the loaded session equal the ones saved, and the session resumes with the
item that was on screen, 39. It ends after 19 items, with the stop reason `stopper`; the five
declines before the restart are the only ones. The restarted process creates its generator
unseeded, as an application does, and the output still repeats: of the strategies here, only the
selector draws from the generator, to break exact ties between items, and none occur.

The [Live CAT sessions](https://douglasrizzo.com.br/catsim/live-sessions.html) guide covers what this notebook leaves out: storing session ids,
sessions that are abandoned, and trackers in several processes. To write a strategy or an exposure
control of your own, see [Extending catsim](https://douglasrizzo.com.br/catsim/extending.html) and the
[custom-strategies notebook](04_custom_strategies.ipynb).